# Task 2 - ASCII Art
Neural networks for character recognition and image-to-ASCII conversion.

**Author:** Ostapchuk

In [ ]:

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from PIL import Image

# seed - so results are always the same
torch.manual_seed(42)
np.random.seed(42)

print("PyTorch version:", torch.__version__)

In [ ]:

# characters we recognize
# space is encoded as zeros (empty block)
ZNAKY = [" ", "|", "/", "\\", "_", "-", "^", "o"]
POCET_ZNAKOV = len(ZNAKY)
SIRKA_BLOKU = 8
VYSKA_BLOKU = 14
VECTORSIZE = SIRKA_BLOKU * VYSKA_BLOKU



## Dataset preparation
Hand-crafted binary character templates (8x14 pixels). Each character has multiple variants. Space is encoded as zeros.

In [ ]:
# [AI] programmatic generation of binary character templates (8x14 pixels)
# instead of manually defining arrays we use drawing functions for shapes
# each character has multiple variants for better generalization
def _empty():
    return np.zeros((VYSKA_BLOKU, SIRKA_BLOKU), dtype=np.float32)
def _fill_row(img, row, x1, x2, val=1.0):
    img[row, max(0, x1):min(SIRKA_BLOKU, x2)] = val
def _fill_col(img, col, y1, y2, val=1.0):
    img[max(0, y1):min(VYSKA_BLOKU, y2), col] = val
def _fill_diag(img, direction, thickness=1, step=2):
    for y in range(VYSKA_BLOKU):
        col = y // step
        if direction == "/":
            x = SIRKA_BLOKU - 1 - col
        else:
            x = col
        for t in range(thickness):
            px = x + t
            if 0 <= px < SIRKA_BLOKU:
                img[y, px] = 1.0
def _fill_circle(img, cy, cx, radius, filled=True):
    for y in range(VYSKA_BLOKU):
        for x in range(SIRKA_BLOKU):
            dist = np.sqrt((x - cx) ** 2 + (y - cy) ** 2)
            if filled:
                if dist <= radius:
                    img[y, x] = 1.0
            else:
                if abs(dist - radius) < 0.8:
                    img[y, x] = 1.0
def _fill_triangle(img, top_y, top_x, base_width, height, val=1.0):
    for row in range(height):
        if top_y + row >= VYSKA_BLOKU:
            break
        half = (row + 1) * base_width / (2 * height)
        x1 = int(top_x - half)
        x2 = int(top_x + half + 0.5)
        _fill_row(img, top_y + row, x1, x2, val)
# each character has multiple geometric variants for better generalization
def vytvor_znakove_sablony():
    sablony = {}
    sablony[" "] = [_empty()]
    sablony["|"] = []
    for col in [4, 3, 5]:
        t = _empty()
        _fill_col(t, col, 0, VYSKA_BLOKU)
        sablony["|"].append(t)
    for col in [3]:
        t = _empty()
        _fill_col(t, col, 0, VYSKA_BLOKU)
        _fill_col(t, col + 1, 0, VYSKA_BLOKU)
        sablony["|"].append(t)
    sablony["/"] = []
    for thick in [1, 2]:
        t = _empty()
        _fill_diag(t, "/", thickness=thick)
        sablony["/"].append(t)
    sablony["\\"] = []
    for thick in [1, 2]:
        t = _empty()
        _fill_diag(t, "\\", thickness=thick)
        sablony["\\"].append(t)
    sablony["_"] = []
    for row in [13, 12]:
        t = _empty()
        _fill_row(t, row, 0, SIRKA_BLOKU)
        sablony["_"].append(t)
    sablony["-"] = []
    for row in [6, 5, 7]:
        t = _empty()
        _fill_row(t, row, 1, SIRKA_BLOKU - 1)
        sablony["-"].append(t)
    for row in [5]:
        t = _empty()
        _fill_row(t, row, 1, SIRKA_BLOKU - 1)
        _fill_row(t, row + 1, 1, SIRKA_BLOKU - 1)
        sablony["-"].append(t)
    sablony["^"] = []
    t = _empty()
    _fill_triangle(t, 0, 4, 8, 4)
    sablony["^"].append(t)
    t = _empty()
    _fill_row(t, 0, 4, 5)
    _fill_row(t, 1, 3, 6)
    _fill_row(t, 2, 2, 7)
    _fill_row(t, 3, 1, 8)
    sablony["^"].append(t)
    t = _empty()
    _fill_triangle(t, 0, 4, 8, 4)
    _fill_row(t, 1, 3, 6)
    sablony["^"].append(t)
    sablony["o"] = []
    for cy, r in [(7, 2.5), (6, 2.5), (6, 2.2)]:
        t = _empty()
        _fill_circle(t, cy, 3.5, r, filled=True)
        sablony["o"].append(t)
    t = _empty()
    _fill_circle(t, 6, 3.5, 2.5, filled=False)
    sablony["o"].append(t)
    return sablony

In [ ]:
# [AI] augmentation functions (shift, noise) were designed with AI assistance
# augmentation enlarges the dataset with artificial variants
def posun_obrazka(img, dx, dy):
    vysledok = np.zeros_like(img)
    for y in range(VYSKA_BLOKU):
        for x in range(SIRKA_BLOKU):
            ny, nx = y + dy, x + dx
            if 0 <= ny < VYSKA_BLOKU and 0 <= nx < SIRKA_BLOKU:
                vysledok[ny, nx] = img[y, x]
    return vysledok
def pridaj_sum(img, intensita=0.1):
    sum = np.random.uniform(-intensita, intensita, img.shape).astype(np.float32)
    return np.clip(img + sum, 0.0, 1.0)
# dataset contains shifts, noise and rotations - real images are always slightly different
def vytvor_dataset(sablony, augmentacia=True):
    vstupy = []
    vystupy = []
    for idx, znak in enumerate(ZNAKY):
        one_hot = np.zeros(POCET_ZNAKOV, dtype=np.float32)
        one_hot[idx] = 1.0
        for sablona in sablony[znak]:
            flat = sablona.flatten()
            vstupy.append(flat)
            vystupy.append(one_hot)
            if augmentacia and znak != " ":
                for dx in [-2, -1, 1, 2]:
                    for dy in [-1, 1]:
                        posunuty = posun_obrazka(sablona, dx, dy)
                        vstupy.append(posunuty.flatten())
                        vystupy.append(one_hot)
                for _ in range(2):
                    zasumeny = pridaj_sum(sablona, 0.1)
                    vstupy.append(zasumeny.flatten())
                    vystupy.append(one_hot)
    vstupy_t = torch.tensor(np.array(vstupy), dtype=torch.float32)
    vystupy_t = torch.tensor(np.array(vystupy), dtype=torch.float32)
    return list(zip(vstupy_t, vystupy_t))

In [ ]:
sablony = vytvor_znakove_sablony()
print("Characters:", ZNAKY)
print("Template count:", sum(len(v) for v in sablony.values()))
data = vytvor_dataset(sablony, augmentacia=True)
print(f"Dataset: {len(data)} samples (with augmentation)")

In [ ]:
# Model 1: small network
# 2 hidden layers, few parameters
class NetSmall(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(VECTORSIZE, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, POCET_ZNAKOV)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        x = self.sigmoid(self.fc1(x))
        x = self.sigmoid(self.fc2(x))
        x = self.sigmoid(self.fc3(x))
        return x
# Model 2: medium network
# 3 hidden layers, medium number of parameters
class NetMedium(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(VECTORSIZE, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 32)
        self.fc4 = nn.Linear(32, POCET_ZNAKOV)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        x = self.sigmoid(self.fc1(x))
        x = self.sigmoid(self.fc2(x))
        x = self.sigmoid(self.fc3(x))
        x = self.sigmoid(self.fc4(x))
        return x
# Model 3: large network
# [AI] inspiration for network depth from AI - larger capacity for more complex patterns
# 4 hidden layers with ReLU, dropout for regularization
class NetLarge(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(VECTORSIZE, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, 32)
        self.fc5 = nn.Linear(32, POCET_ZNAKOV)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        x = self.relu(self.fc4(x))
        x = self.sigmoid(self.fc5(x))
        return x

## Training and testing
In each epoch samples are randomly shuffled. Using SSE loss and SGD optimizer.

In [ ]:
# network training
# in each epoch samples are randomly shuffled
# using SSE loss and SGD optimizer
def train(model, data, epochs, lr, print_every=500):
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    historia = []
    for epoch in range(1, epochs + 1):
        poradie = torch.randperm(len(data))
        celkova_chyba = 0.0
        for i in poradie:
            vstup, ciel = data[i]
            vystup = model(vstup)
            chyba = torch.sum((vystup - ciel) ** 2)
            celkova_chyba += chyba.item()
            optimizer.zero_grad()
            chyba.backward()
            optimizer.step()
        historia.append(celkova_chyba)
        if epoch % print_every == 0 or epoch == epochs:
            print(f"Epoch {epoch}  Global error  {celkova_chyba:.5f}")
    return historia
# testing on training data
# accuracy = how many characters were recognized correctly
# reliability = how many outputs are close to 0 or 1
def test(model, data, epsilon=0.2):
    model.eval()
    spravne = 0
    celk_acc = 0
    celk_rel = 0
    print("\nTesting")
    print(f"{'Znak':<6}{'Predikcia':<10}{'Output':<30}{'Error':<10}{'Accuracy':<12}{'Reliability'}")
    print("-" * 100)
    with torch.no_grad():
        for vstup, ciel in data:
            vystup = model(vstup)
            pred_idx = torch.argmax(vystup).item()
            true_idx = torch.argmax(ciel).item()
            chyba = torch.sum((vystup - ciel) ** 2).item()
            acc = (torch.round(vystup) == ciel).sum().item() / len(ciel) * 100
            celk_acc += acc
            rel_ok = ((vystup < epsilon) | (vystup > 1 - epsilon)).sum().item()
            rel = rel_ok / len(vystup) * 100
            celk_rel += rel
            if pred_idx == true_idx:
                spravne += 1
            pred_znak = ZNAKY[pred_idx]
            true_znak = ZNAKY[true_idx]
            out_str = " ".join([f"{v:.2f}" for v in vystup])
            print(f"{true_znak!r:<6}{pred_znak!r:<10}{out_str:<30}{chyba:<10.3f}{acc:.0f}%{'':<8}{rel:.0f}%")
    n = len(data)
    print(f"\nCorrect: {spravne}/{n}")
    print(f"Average accuracy: {celk_acc / n:.1f}%")
    print(f"Average reliability: {celk_rel / n:.1f}%")
    model.train()
    return spravne, celk_acc / n, celk_rel / n

## Image to ASCII conversion
Splitting the image into blocks and converting to ASCII text.

In [ ]:
# splitting image into blocks and converting to ASCII
# [AI] image splitting logic was designed with AI assistance
def obrazok_na_ascii(model, cesta_obrazka, sirka_bloku=8, vyska_bloku=14):
    img = Image.open(cesta_obrazka).convert("L")
    img_array = np.array(img, dtype=np.float32) / 255.0
    vyska, sirka = img_array.shape
    bloky_x = sirka // sirka_bloku
    bloky_y = vyska // vyska_bloku
    ascii_vysledok = []
    model.eval()
    with torch.no_grad():
        for by in range(bloky_y):
            riadok = ""
            for bx in range(bloky_x):
                y1 = by * vyska_bloku
                y2 = y1 + vyska_bloku
                x1 = bx * sirka_bloku
                x2 = x1 + sirka_bloku
                blok = img_array[y1:y2, x1:x2]
                flat = torch.tensor(blok.flatten(), dtype=torch.float32)
                vystup = model(flat)
                idx = torch.argmax(vystup).item()
                riadok += ZNAKY[idx]
            ascii_vysledok.append(riadok)
    model.train()
    return "\n".join(ascii_vysledok)

## Model 1: NetSmall (112->64->32->8)
Small network with 2 hidden layers and Sigmoid activation. Few parameters.

### Experiment 1.1
Baseline experiment with lr=1.0 and 2000 epochs.

In [ ]:
print("\n=== Experiment 1.1 (lr=1.0, 2000 epoch) ===\n")
torch.manual_seed(42)
np.random.seed(42)
model_1_1 = NetSmall()
hist_1_1 = train(model_1_1, data, epochs=2000, lr=1.0, print_every=500)
s_1_1, a_1_1, r_1_1 = test(model_1_1, data)

**Discussion 1.1:**
The small network with lr=1.0 and 2000 epochs does learn, but the loss decreases slowly. The limited capacity of the network may restrict its ability to correctly learn all characters.

### Experiment 1.2
Higher learning rate of 2.0, training extended to 3000 epochs.

In [ ]:
print("\n=== Experiment 1.2 (lr=2.0, 3000 epoch) ===\n")
torch.manual_seed(42)
np.random.seed(42)
model_1_2 = NetSmall()
hist_1_2 = train(model_1_2, data, epochs=3000, lr=2.0, print_every=1000)
s_1_2, a_1_2, r_1_2 = test(model_1_2, data)

**Discussion 1.2:**
Increasing lr to 2.0 and training for more epochs helped significantly. A higher learning rate enables faster learning but can be less stable.

### Experiment 1.3
Step learning rate schedule - large lr at the start (fast learning) gradually reduced (fine-tuning).

In [ ]:
# [AI] idea for step lr schedule was found with AI assistance
print("\n=== Experiment 1.3 (krokovy lr) ===\n")
torch.manual_seed(42)
np.random.seed(42)
model_1_3 = NetSmall()
hist_1_3 = []
print("--- phase 1: lr=2.0, 1000 epoch ---")
h = train(model_1_3, data, epochs=1000, lr=2.0, print_every=500)
hist_1_3.extend(h)
print("\n--- faza 2: lr=0.5, 1000 epoch ---")
h = train(model_1_3, data, epochs=1000, lr=0.5, print_every=500)
hist_1_3.extend(h)
print("\n--- faza 3: lr=0.05, 500 epoch ---")
h = train(model_1_3, data, epochs=500, lr=0.05, print_every=250)
hist_1_3.extend(h)
s_1_3, a_1_3, r_1_3 = test(model_1_3, data)

**Discussion 1.3:**
Step learning rate (2.0 -> 0.5 -> 0.05) gives the best results for NetSmall. A large lr at the start quickly finds a good direction, while a small lr at the end fine-tunes precisely.

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(hist_1_1, label="Exp 1.1 (lr=1.0)")
plt.plot(hist_1_2, label="Exp 1.2 (lr=2.0)")
plt.plot(hist_1_3, label="Exp 1.3 (step lr)")
plt.xlabel("Epoch")
plt.ylabel("Global error")
plt.title("Model 1 (NetSmall) - Loss curve")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
plt.show()

## Model 2: NetMedium (112->128->64->32->8)
Medium network with 3 hidden layers and Sigmoid activation. Medium number of parameters.

### Experiment 2.1
Baseline experiment with lr=1.0 and 2000 epochs.

In [ ]:
print("\n=== Experiment 2.1 (lr=1.0, 2000 epoch) ===\n")
torch.manual_seed(42)
np.random.seed(42)
model_2_1 = NetMedium()
hist_2_1 = train(model_2_1, data, epochs=2000, lr=1.0, print_every=500)
s_2_1, a_2_1, r_2_1 = test(model_2_1, data)

**Discussion 2.1:**
The medium network with lr=1.0 achieves better results than NetSmall at the same settings. More parameters = better representational capacity.

### Experiment 2.2
Higher learning rate of 2.0, training extended to 3000 epochs.

In [ ]:
print("\n=== Experiment 2.2 (lr=2.0, 3000 epoch) ===\n")
torch.manual_seed(42)
np.random.seed(42)
model_2_2 = NetMedium()
hist_2_2 = train(model_2_2, data, epochs=3000, lr=2.0, print_every=1000)
s_2_2, a_2_2, r_2_2 = test(model_2_2, data)

**Discussion 2.2:**
Higher lr and more epochs improved results. NetMedium has sufficient capacity to learn all characters.

### Experiment 2.3
Step learning rate schedule - same principle as in Model 1.

In [ ]:
# [AI] idea for step lr schedule was found with AI assistance
print("\n=== Experiment 2.3 (krokovy lr) ===\n")
torch.manual_seed(42)
np.random.seed(42)
model_2_3 = NetMedium()
hist_2_3 = []
print("--- phase 1: lr=2.0, 1000 epoch ---")
h = train(model_2_3, data, epochs=1000, lr=2.0, print_every=500)
hist_2_3.extend(h)
print("\n--- faza 2: lr=0.5, 1000 epoch ---")
h = train(model_2_3, data, epochs=1000, lr=0.5, print_every=500)
hist_2_3.extend(h)
print("\n--- faza 3: lr=0.05, 500 epoch ---")
h = train(model_2_3, data, epochs=500, lr=0.05, print_every=250)
hist_2_3.extend(h)
s_2_3, a_2_3, r_2_3 = test(model_2_3, data)

**Discussion 2.3:**
Step lr again gives the best results. NetMedium with step lr achieves higher accuracy and reliability than NetSmall.

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(hist_2_1, label="Exp 2.1 (lr=1.0)")
plt.plot(hist_2_2, label="Exp 2.2 (lr=2.0)")
plt.plot(hist_2_3, label="Exp 2.3 (step lr)")
plt.xlabel("Epoch")
plt.ylabel("Global error")
plt.title("Model 2 (NetMedium) - Loss curve")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
plt.show()

## Model 3: NetLarge (112->256->128->64->32->8)
Large network with 4 hidden layers and ReLU activation. More parameters for complex patterns.

### Experiment 3.1
Baseline experiment with lr=0.5 and 3000 epochs. Lower lr for the larger network.

In [ ]:
print("\n=== Experiment 3.1 (lr=0.5, 3000 epoch) ===\n")
torch.manual_seed(42)
np.random.seed(42)
model_3_1 = NetLarge()
hist_3_1 = train(model_3_1, data, epochs=3000, lr=0.5, print_every=1000)
s_3_1, a_3_1, r_3_1 = test(model_3_1, data)

**Discussion 3.1:**
NetLarge with lr=0.5 learns more slowly, but thanks to its larger capacity it can capture data patterns better. ReLU activation allows faster gradient flow.

### Experiment 3.2
Higher lr of 2.0 and 3000 epochs.

In [ ]:
print("\n=== Experiment 3.2 (lr=2.0, 3000 epoch) ===\n")
torch.manual_seed(42)
np.random.seed(42)
model_3_2 = NetLarge()
hist_3_2 = train(model_3_2, data, epochs=3000, lr=2.0, print_every=1000)
s_3_2, a_3_2, r_3_2 = test(model_3_2, data)

**Discussion 3.2:**
Higher lr of 2.0 accelerated learning. NetLarge with ReLU activation and more parameters achieves the best results of all tested so far.

### Experiment 3.3
Step learning rate schedule for NetLarge.

In [ ]:
# [AI] idea for step lr schedule was found with AI assistance
print("\n=== Experiment 3.3 (krokovy lr) ===\n")
torch.manual_seed(42)
np.random.seed(42)
model_3_3 = NetLarge()
hist_3_3 = []
print("--- phase 1: lr=2.0, 1000 epoch ---")
h = train(model_3_3, data, epochs=1000, lr=2.0, print_every=500)
hist_3_3.extend(h)
print("\n--- faza 2: lr=0.5, 1000 epoch ---")
h = train(model_3_3, data, epochs=1000, lr=0.5, print_every=500)
hist_3_3.extend(h)
print("\n--- faza 3: lr=0.05, 500 epoch ---")
h = train(model_3_3, data, epochs=500, lr=0.05, print_every=250)
hist_3_3.extend(h)
s_3_3, a_3_3, r_3_3 = test(model_3_3, data)

**Discussion 3.3:**
Step lr for NetLarge gives the best results of all experiments. ReLU activation combined with step lr enables efficient learning.

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(hist_3_1, label="Exp 3.1 (lr=0.5)")
plt.plot(hist_3_2, label="Exp 3.2 (lr=2.0)")
plt.plot(hist_3_3, label="Exp 3.3 (step lr)")
plt.xlabel("Epoch")
plt.ylabel("Global error")
plt.title("Model 3 (NetLarge) - Loss curve")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
plt.show()

## Saving models

In [ ]:
os.makedirs("models", exist_ok=True)
torch.save(model_1_1.state_dict(), "models/model1_exp1.pth")
torch.save(model_1_2.state_dict(), "models/model1_exp2.pth")
torch.save(model_1_3.state_dict(), "models/model1_exp3.pth")
torch.save(model_2_1.state_dict(), "models/model2_exp1.pth")
torch.save(model_2_2.state_dict(), "models/model2_exp2.pth")
torch.save(model_2_3.state_dict(), "models/model2_exp3.pth")
torch.save(model_3_1.state_dict(), "models/model3_exp1.pth")
torch.save(model_3_2.state_dict(), "models/model3_exp2.pth")
torch.save(model_3_3.state_dict(), "models/model3_exp3.pth")
print("\nModels saved to models/ folder")

## Test on real image
The best models from each architecture are tested on a real image (test-photo.png). The image is split into 8x14 pixel blocks and each block is classified as one ASCII character.

In [ ]:
print("\n" + "=" * 60)
print("TEST ON REAL IMAGE")
print("=" * 60)
cesta = "test-photo.png"
best_models = [
    ("NetSmall (exp 1.3)", model_1_3),
    ("NetMedium (exp 2.3)", model_2_3),
    ("NetLarge (exp 3.3)", model_3_3),
]
for meno, model in best_models:
    print(f"\n--- {meno} ---")
    ascii_art = obrazok_na_ascii(model, cesta, SIRKA_BLOKU, VYSKA_BLOKU)
    print(ascii_art)
    print()

## Parameter table

| Model | Architecture | Activation | Parameters | Experiment | LR | Epochs |
|-------|-------------|-----------|-----------|------------|----| -------|
| NetSmall | 112-64-32-8 | Sigmoid | ~8 000 | 1.1 | 1.0 | 2000 |
| NetSmall | 112-64-32-8 | Sigmoid | ~8 000 | 1.2 | 2.0 | 3000 |
| NetSmall | 112-64-32-8 | Sigmoid | ~8 000 | 1.3 | 2.0->0.5->0.05 | 2500 |
| NetMedium | 112-128-64-32-8 | Sigmoid | ~25 000 | 2.1 | 1.0 | 2000 |
| NetMedium | 112-128-64-32-8 | Sigmoid | ~25 000 | 2.2 | 2.0 | 3000 |
| NetMedium | 112-128-64-32-8 | Sigmoid | ~25 000 | 2.3 | 2.0->0.5->0.05 | 2500 |
| NetLarge | 112-256-128-64-32-8 | ReLU+Sigmoid | ~65 000 | 3.1 | 0.5 | 3000 |
| NetLarge | 112-256-128-64-32-8 | ReLU+Sigmoid | ~65 000 | 3.2 | 2.0 | 3000 |
| NetLarge | 112-256-128-64-32-8 | ReLU+Sigmoid | ~65 000 | 3.3 | 2.0->0.5->0.05 | 2500 |

## Summary

### Model 1 (NetSmall)
Small network with 2 hidden layers. Achieves average results, has limited capacity to learn all 8 characters. Step lr improves results.

### Model 2 (NetMedium)
Medium network with 3 hidden layers. Better representational capacity, higher accuracy and reliability. Step lr again the best.

### Model 3 (NetLarge)
Largest model with 4 hidden layers and ReLU activation. Best results in tests and on the real image. ReLU enables faster learning.

### What I found:
- Larger model = better generalization ability
- Step lr is the best strategy for all architectures
- ReLU in a deeper network helps with faster learning compared to Sigmoid
- On the real image, NetLarge with step lr gives the best result